# Phase 1 — Dataset Setup

Run this notebook **first** to unzip the dataset, validate all files,
and ensure everything is ready for feature engineering and modeling.

**Runtime**: ~1 minute

## 1. Setup

In [1]:
import os, sys

# Ensure we're in the notebook's directory (participant_kit/phase1/)
_this_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in dir() else None
if _this_dir is None:
    for _candidate in [os.getcwd(), os.path.join(os.getcwd(), "participant_kit", "phase1")]:
        if os.path.exists(os.path.join(_candidate, "utils.py")):
            os.chdir(_candidate)
            break
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

import zipfile
from pathlib import Path

import pandas as pd
import json

In [ ]:
## 2. Unzip Dataset

In [2]:
ZIP_PATH = Path("phase1_dataset.zip")  # adjust if you placed it elsewhere
DEST = Path("phase1_dataset")

if DEST.exists():
    print(f"Dataset directory already exists: {DEST.resolve()}")
    print("Skipping extraction. Delete the directory to re-extract.")
else:
    assert ZIP_PATH.exists(), (
        f"Dataset ZIP not found at: {ZIP_PATH.resolve()}\n"
        f"Place phase1_dataset.zip in this directory ({Path('.').resolve()}) and re-run."
    )
    print(f"Extracting {ZIP_PATH} ...")
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall()
    print(f"Extracted to {DEST.resolve()}")

Dataset directory already exists: C:\Pythonwork\Hackathon-Sea-Winds-Predictions\phase_1\phase1_dataset
Skipping extraction. Delete the directory to re-extract.


## 3. Configuration

In [3]:
DATA_DIR = DEST
TRAIN_DIR = DATA_DIR / "train"
INFERENCE_DIR = DATA_DIR / "inference"
SCORING_DIR = DATA_DIR / "scoring"
FEATURES_DIR = DATA_DIR / "features"

REGIONS = ["north_sea", "east_china_sea"]
NUM_WINDOWS = 8


def _fmt_size(nbytes):
    """Format byte count as a human-readable string."""
    if nbytes >= 1 << 30:
        return f"{nbytes / (1 << 30):.2f} GB"
    if nbytes >= 1 << 20:
        return f"{nbytes / (1 << 20):.1f} MB"
    if nbytes >= 1 << 10:
        return f"{nbytes / (1 << 10):.1f} KB"
    return f"{nbytes} B"

## 4. Validate Directory Structure

In [4]:
expected_dirs = {
    "train": TRAIN_DIR,
    "inference": INFERENCE_DIR,
    "scoring": SCORING_DIR,
}
# Inference sub-directories
for w in range(1, NUM_WINDOWS + 1):
    expected_dirs[f"inference/window_{w}"] = INFERENCE_DIR / f"window_{w}"

dir_ok = True
for name, path in expected_dirs.items():
    exists = path.is_dir()
    status = "OK" if exists else "MISSING"
    if not exists:
        dir_ok = False
    print(f"  [{status:>7s}]  {name}/")

# features/ may not exist yet — that's fine, it's created by feature engineering
if FEATURES_DIR.is_dir():
    print(f"  [     OK]  features/  (already created)")
else:
    print(f"  [   INFO]  features/  (will be created by 1_feature_engineering.ipynb)")

if dir_ok:
    print("\nDirectory structure: OK")
else:
    print("\nWARNING: Some directories are missing. Re-extract the dataset.")


  [     OK]  train/
  [     OK]  inference/
  [     OK]  scoring/
  [     OK]  inference/window_1/
  [     OK]  inference/window_2/
  [     OK]  inference/window_3/
  [     OK]  inference/window_4/
  [     OK]  inference/window_5/
  [     OK]  inference/window_6/
  [     OK]  inference/window_7/
  [     OK]  inference/window_8/
  [   INFO]  features/  (will be created by 1_feature_engineering.ipynb)

Directory structure: OK


## 5. Validate Training Files

In [5]:
# Build the expected file list for train/
train_files = []
for region in REGIONS:
    train_files.extend([
        f"reanalysis_{region}_6h.parquet",
        f"reanalysis_pressure_{region}.parquet",
        f"stations_{region}_6h.parquet",
        f"hres_{region}.parquet",
        f"hres_pressure_{region}.parquet",
        f"elevation_{region}.nc",
        f"lsm_{region}.nc",
    ])
# Worldwide files
for year in range(2019, 2022):  # 2022 worldwide is shipped per-window (see inference/window_*/)
    train_files.append(f"reanalysis_worldwide_daily_{year}.parquet")
train_files.append("elevation_worldwide_5arcmin.nc")

rows = []
total_bytes = 0
all_present = True

for fname in train_files:
    fpath = TRAIN_DIR / fname
    exists = fpath.exists()
    if exists:
        size = fpath.stat().st_size
        total_bytes += size
        rows.append({"File": fname, "Status": "OK", "Size": _fmt_size(size)})
    else:
        all_present = False
        rows.append({"File": fname, "Status": "MISSING", "Size": "-"})

df_train = pd.DataFrame(rows)
display(df_train.style.hide(axis="index"))

n_found = (df_train["Status"] == "OK").sum()
print(f"\nTraining files: {n_found}/{len(train_files)} present  ({_fmt_size(total_bytes)} total)")
if not all_present:
    print("WARNING: Some training files are missing!")

File,Status,Size
reanalysis_north_sea_6h.parquet,OK,1.22 GB
reanalysis_pressure_north_sea.parquet,OK,475.9 MB
stations_north_sea_6h.parquet,OK,241.9 KB
hres_north_sea.parquet,OK,296.5 MB
hres_pressure_north_sea.parquet,OK,1.41 GB
elevation_north_sea.nc,OK,33.7 KB
lsm_north_sea.nc,OK,29.5 KB
reanalysis_east_china_sea_6h.parquet,OK,1.22 GB
reanalysis_pressure_east_china_sea.parquet,OK,473.3 MB
stations_east_china_sea_6h.parquet,OK,268.6 KB



Training files: 18/18 present  (8.57 GB total)


## 6. Validate Inference Windows

In [6]:
window_expected_files = ["metadata.json", "context_worldwide_daily.parquet"]
for region in REGIONS:
    window_expected_files.extend([
        f"context_reanalysis_{region}.parquet",
        f"context_reanalysis_pressure_{region}.parquet",
        f"context_stations_{region}.parquet",
        f"context_hres_{region}.parquet",
        f"context_hres_pressure_{region}.parquet",
    ])

complete_windows = []
incomplete_windows = []

for w in range(1, NUM_WINDOWS + 1):
    win_dir = INFERENCE_DIR / f"window_{w}"
    if not win_dir.is_dir():
        incomplete_windows.append((w, "directory missing"))
        continue

    missing = [f for f in window_expected_files if not (win_dir / f).exists()]
    if missing:
        incomplete_windows.append((w, f"missing {len(missing)} file(s): {', '.join(missing)}"))
    else:
        # Read metadata for summary
        meta = json.loads((win_dir / "metadata.json").read_text())
        complete_windows.append(meta)

print(f"Inference windows: {len(complete_windows)}/{NUM_WINDOWS} complete")
print()

if complete_windows:
    rows = []
    for m in complete_windows:
        rows.append({
            "Window": m["id"],
            "Context": f"{m['context_start']} to {m['context_end']}",
            "Predict": f"{m['predict_start']} to {m['predict_end']}",
            "d1": m["score_days"]["d1"],
            "d7": m["score_days"]["d7"],
            "d14": m["score_days"]["d14"],
        })
    display(pd.DataFrame(rows).style.hide(axis="index"))

if incomplete_windows:
    print("\nIncomplete windows:")
    for w, reason in incomplete_windows:
        print(f"  Window {w}: {reason}")

Inference windows: 8/8 complete



Window,Context,Predict,d1,d7,d14
1,2022-01-01 to 2022-01-14,2022-01-15 to 2022-01-28,2022-01-15,2022-01-21,2022-01-28
2,2022-02-12 to 2022-02-25,2022-02-26 to 2022-03-11,2022-02-26,2022-03-04,2022-03-11
3,2022-03-26 to 2022-04-08,2022-04-09 to 2022-04-22,2022-04-09,2022-04-15,2022-04-22
4,2022-05-07 to 2022-05-20,2022-05-21 to 2022-06-03,2022-05-21,2022-05-27,2022-06-03
5,2022-06-18 to 2022-07-01,2022-07-02 to 2022-07-15,2022-07-02,2022-07-08,2022-07-15
6,2022-07-30 to 2022-08-12,2022-08-13 to 2022-08-26,2022-08-13,2022-08-19,2022-08-26
7,2022-09-10 to 2022-09-23,2022-09-24 to 2022-10-07,2022-09-24,2022-09-30,2022-10-07
8,2022-10-22 to 2022-11-04,2022-11-05 to 2022-11-18,2022-11-05,2022-11-11,2022-11-18


## 7. Validate Scoring Files

In [7]:
scoring_files = [
    "sample_submission.csv",
    "submission_format.md",
    "station_metadata.csv",
]

rows = []
for fname in scoring_files:
    fpath = SCORING_DIR / fname
    exists = fpath.exists()
    size = _fmt_size(fpath.stat().st_size) if exists else "-"
    rows.append({"File": fname, "Status": "OK" if exists else "MISSING", "Size": size})

df_scoring = pd.DataFrame(rows)
display(df_scoring.style.hide(axis="index"))

n_found = (df_scoring["Status"] == "OK").sum()
print(f"\nScoring files: {n_found}/{len(scoring_files)} present")


File,Status,Size
sample_submission.csv,OK,7.4 KB
submission_format.md,OK,5.1 KB
station_metadata.csv,OK,964 B



Scoring files: 3/3 present


## 8. Quick Data Preview

Load one training parquet to verify the data is readable and inspect its structure.

In [8]:
preview_file = TRAIN_DIR / f"reanalysis_{REGIONS[0]}_6h.parquet"

if preview_file.exists():
    print(f"Loading {preview_file.name} ...")
    df = pd.read_parquet(preview_file)

    print(f"\nShape: {df.shape[0]:,} rows x {df.shape[1]} columns")
    print(f"Columns: {list(df.columns)}")

    # Time range
    if "time" in df.columns:
        print(f"\nTime range: {df['time'].min()} to {df['time'].max()}")

    # Grid points
    if "latitude" in df.columns and "longitude" in df.columns:
        n_lat = df["latitude"].nunique()
        n_lon = df["longitude"].nunique()
        print(f"Grid: {n_lat} latitudes x {n_lon} longitudes = {n_lat * n_lon} grid points")
        print(f"Latitude range:  {df['latitude'].min():.2f} to {df['latitude'].max():.2f}")
        print(f"Longitude range: {df['longitude'].min():.2f} to {df['longitude'].max():.2f}")

    print(f"\nMemory usage: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
else:
    print(f"Preview file not found: {preview_file}")
    print("Skipping data preview.")
    df = None

Loading reanalysis_north_sea_6h.parquet ...

Shape: 11,244,960 rows x 39 columns
Columns: ['time', 'latitude', 'longitude', 'u10', 'v10', 'u100', 'v100', 't2m', 'msl', 'sshf', 'z700', 'z850', 'sst', 'blh', 'cape', 'speed_d1_h0', 'dir_d1_h0', 'speed_d1_h6', 'dir_d1_h6', 'speed_d1_h12', 'dir_d1_h12', 'speed_d1_h18', 'dir_d1_h18', 'speed_d7_h0', 'dir_d7_h0', 'speed_d7_h6', 'dir_d7_h6', 'speed_d7_h12', 'dir_d7_h12', 'speed_d7_h18', 'dir_d7_h18', 'speed_d14_h0', 'dir_d14_h0', 'speed_d14_h6', 'dir_d14_h6', 'speed_d14_h12', 'dir_d14_h12', 'speed_d14_h18', 'dir_d14_h18']

Time range: 2019-01-01 00:00:00 to 2021-12-31 18:00:00
Grid: 45 latitudes x 57 longitudes = 2565 grid points
Latitude range:  51.00 to 62.00
Longitude range: -4.00 to 10.00

Memory usage: 1799.2 MB


In [9]:
if df is not None:
    print("First 3 rows:")
    display(df.head(3))

    print("\nNaN summary (columns with any missing values):")
    nan_counts = df.isna().sum()
    nan_cols = nan_counts[nan_counts > 0]
    if len(nan_cols) == 0:
        print("  No missing values.")
    else:
        nan_pct = (nan_cols / len(df) * 100).round(1)
        nan_df = pd.DataFrame({"NaN Count": nan_cols, "NaN %": nan_pct})
        display(nan_df)

    del df  # free memory

First 3 rows:


,time,latitude,longitude,u10,v10,u100,v100,t2m,msl,sshf,...,speed_d7_h18,dir_d7_h18,speed_d14_h0,dir_d14_h0,speed_d14_h6,dir_d14_h6,speed_d14_h12,dir_d14_h12,speed_d14_h18,dir_d14_h18
0,2019-01-01 00:00:00,51.0,-4.0,3.476440,-1.635300,5.654449,-3.095764,282.117188,103686.4375,52072.0,...,4.676239,343.706757,3.435754,260.92804,4.670116,242.1633,6.10004,252.821442,5.280574,222.165863
1,2019-01-01 06:00:00,51.0,-4.0,2.874985,-2.598663,4.587036,-4.446609,282.357178,103660.6250,50027.0,...,4.676239,343.706757,3.435754,260.92804,4.670116,242.1633,6.10004,252.821442,5.280574,222.165863
2,2019-01-01 12:00:00,51.0,-4.0,3.110641,-3.346024,4.245331,-4.623001,282.512207,103759.5000,-55619.0,...,4.676239,343.706757,3.435754,260.92804,4.670116,242.1633,6.10004,252.821442,5.280574,222.165863



NaN summary (columns with any missing values):


,NaN Count,NaN %
sst,4327008,38.5


## 9. Sample Submission Preview

In [10]:
sample_path = SCORING_DIR / "sample_submission.csv"

if sample_path.exists():
    sample = pd.read_csv(sample_path)
    print(f"Sample submission: {sample.shape[0]:,} rows x {sample.shape[1]} columns")
    print(f"Columns: {list(sample.columns)}")
    print(f"\nWindows: {sorted(sample['window'].unique())}")
    print(f"Regions: {sorted(sample['region'].unique())}")
    print(f"Horizons: {sorted(sample['horizon'].unique())}")
    print(f"Hours: {sorted(sample['hour'].unique())}")
    if 'level' in sample.columns:
        print(f"Levels: {sorted(sample['level'].unique())}")
    print()
    display(sample.head(5))
    del sample
else:
    print(f"sample_submission.csv not found in {SCORING_DIR}")

Sample submission: 144 rows x 11 columns
Columns: ['window', 'region', 'latitude', 'longitude', 'horizon', 'hour', 'level', 'q05', 'q50', 'q95', 'dir_50']

Windows: [np.int64(1)]
Regions: ['east_china_sea', 'north_sea']
Horizons: [np.int64(1), np.int64(7), np.int64(14)]
Hours: [np.int64(0), np.int64(6), np.int64(12), np.int64(18)]
Levels: ['1000', '10m', '500', '700', '850', '925']



,window,region,latitude,longitude,horizon,hour,level,q05,q50,q95,dir_50
0,1,north_sea,56.0,3.0,1,0,10m,3.2,5.3,8.5,225.0
1,1,north_sea,56.0,3.0,1,0,1000,3.2,5.3,8.5,225.0
2,1,north_sea,56.0,3.0,1,0,925,3.2,5.3,8.5,225.0
3,1,north_sea,56.0,3.0,1,0,850,3.2,5.3,8.5,225.0
4,1,north_sea,56.0,3.0,1,0,700,3.2,5.3,8.5,225.0


## 10. Summary

In [11]:
# Collect final status
issues = []

# Check directory structure
if not dir_ok:
    issues.append("Some directories are missing")

# Check training files
n_train_ok = (df_train["Status"] == "OK").sum()
n_train_total = len(train_files)
if n_train_ok < n_train_total:
    issues.append(f"Training files: {n_train_ok}/{n_train_total} present")

# Check inference windows
n_win_ok = len(complete_windows)
if n_win_ok < NUM_WINDOWS:
    issues.append(f"Inference windows: {n_win_ok}/{NUM_WINDOWS} complete")

# Check scoring
n_scoring_ok = (df_scoring["Status"] == "OK").sum()
n_scoring_total = len(scoring_files)
if n_scoring_ok < n_scoring_total:
    issues.append(f"Scoring files: {n_scoring_ok}/{n_scoring_total} present")

print("=" * 55)
print("  DATASET SETUP SUMMARY")
print("=" * 55)

if not issues:
    print(f"  [OK] Dataset extracted")
    print(f"  [OK] Training data: {n_train_total} files ({_fmt_size(total_bytes)} total)")
    print(f"  [OK] Inference windows: {NUM_WINDOWS}/{NUM_WINDOWS} complete")
    print(f"  [OK] Scoring files: {n_scoring_total}/{n_scoring_total} present")
    print()
    print("  All checks passed. Ready to run 1_feature_engineering.ipynb")
else:
    for issue in issues:
        print(f"  [!!] {issue}")
    print()
    print("  Fix the issues above before proceeding.")

print("=" * 55)


  DATASET SETUP SUMMARY
  [OK] Dataset extracted
  [OK] Training data: 18 files (8.57 GB total)
  [OK] Inference windows: 8/8 complete
  [OK] Scoring files: 3/3 present

  All checks passed. Ready to run 1_feature_engineering.ipynb
